# W&B Models Hands-on: One Run

하나의 실험을 하나의 W&B run으로 기록하면서 **config, metrics, summary, alert**를 한 번에 익힙니다.

| 항목 | 내용 |
|---|---|
| 예상 시간 | 10–15분 |
| 필수 | W&B account |
| 생성되는 run | 정확히 1개 |
| 외부 ML framework | 없음 |
| alert | 기본 비활성화, 확인 후 선택적으로 1회 전송 |

## 흐름

1. W&B 설치와 로그인
2. 하나의 run 시작
3. metric과 summary 기록
4. alert 조건 확인
5. run 종료 후 W&B App 확인

---

## 실행 전 확인

`WANDB_API_KEY`를 환경변수로 설정하거나 실행 중 나타나는 `wandb.login()` 프롬프트를 사용합니다. Project를 바꾸려면 `WANDB_PROJECT`를 환경변수로 설정하세요. API key를 notebook에 직접 작성하지 마세요.

아래 코드 셀 하나가 전체 실습입니다. 위에서 아래로 한 번 실행하세요. 셀을 다시 실행하면 새 run이 하나 더 생성됩니다.

실제 알림을 받으려면 W&B **User Settings → Alerts**에서 **Scriptable run alerts**와 email 또는 Slack을 설정한 뒤 `SEND_ALERT = True`로 변경합니다. 처음에는 `False`로 실행해 조건과 메시지만 확인하세요.

In [ ]:
# 0. W&B SDK를 현재 notebook 환경에 설치합니다.
%pip install -qU wandb

# 1. W&B에 로그인하고 run을 저장할 project를 정합니다.
# 환경변수에 API key가 있으면 사용하고, 없으면 로그인 프롬프트를 엽니다.
import os

import wandb
from wandb import AlertLevel

api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
else:
    wandb.login()

PROJECT = os.environ.get("WANDB_PROJECT", "wandb-models-handson")
print("Project:", PROJECT)

# 2. 하나의 run을 시작합니다.
# config에는 learning rate처럼 실행 중 변하지 않는 실험 조건을 기록합니다.
run = wandb.init(
    project=PROJECT,
    name="one-run-handson",
    config={
        "learning_rate": 0.03,
        "epochs": 5,
        "model": "tiny-simulator",
    },
    tags=["handson", "one-run"],
)
print("Run URL:", run.url)

# 3. 매 epoch의 metric을 기록합니다.
# 이 예제는 가상 학습 결과를 사용하므로 실제 코드에서는 아래 값만 교체하면 됩니다.
loss_history = [0.88, 0.67, 0.52, 0.41, 0.35]
accuracy_history = [0.51, 0.60, 0.66, 0.71, 0.74]

for epoch, (loss, accuracy) in enumerate(
    zip(loss_history, accuracy_history),
    start=1,
):
    run.log({
        "epoch": epoch,
        "train/loss": loss,
        "eval/accuracy": accuracy,
    })
    print(f"epoch={epoch} loss={loss:.2f} accuracy={accuracy:.2f}")

# 4. run을 정렬하고 비교할 최종값을 summary에 기록합니다.
accuracy_threshold = 0.75
final_accuracy = accuracy_history[-1]
best_accuracy = max(accuracy_history)

run.summary["final/eval_accuracy"] = final_accuracy
run.summary["best/eval_accuracy"] = best_accuracy
run.summary["final/train_loss"] = loss_history[-1]

# 5. 최종 accuracy가 기준보다 낮을 때만 alert 후보를 만듭니다.
# 기본값은 dry run입니다. W&B Alerts 설정을 마친 뒤 True로 바꾸면 실제 알림이 전송됩니다.
SEND_ALERT = False

if final_accuracy < accuracy_threshold:
    message = (
        f"Final accuracy {final_accuracy:.2f} is below "
        f"threshold {accuracy_threshold:.2f}"
    )
    print("Alert condition met:", message)

    if SEND_ALERT:
        run.alert(
            title="Low evaluation accuracy",
            text=message,
            level=AlertLevel.WARN,
            wait_duration=300,
        )
        print("Alert sent")
    else:
        print("Dry run: set SEND_ALERT=True after configuring Alerts.")

# 6. 기록을 모두 전송하고 run을 완료 상태로 종료합니다.
run_url = run.url
run.finish()
print("Finished one run:", run_url)

---

## W&B App에서 확인할 것

- [ ] Runs table에 `one-run-handson` run이 **1개** 있다.
- [ ] Config에서 learning rate, epochs, model을 볼 수 있다.
- [ ] Charts에서 `train/loss`와 `eval/accuracy` 변화를 볼 수 있다.
- [ ] Overview 또는 Summary에서 final/best accuracy를 볼 수 있다.
- [ ] 기본 실행에서는 alert 메시지만 출력되고 실제 알림은 전송되지 않았다.

다음에는 가상 `loss_history`와 `accuracy_history`를 자신의 실제 학습 metric으로 교체해 보세요.

## 참고 자료

- 구성 참고: [W&B Examples — Intro to Weights & Biases](https://github.com/wandb/examples/blob/master/colabs/intro/Intro_to_Weights_%26_Biases.ipynb)
- Alert 공식 문서: [Send an alert](https://docs.wandb.ai/models/runs/alert)

이 notebook은 공식 Intro의 install/login, run initialization, config, metric logging, finish 흐름을 참고해 단일 run 실습으로 새로 구성했습니다.